In [1]:
import csv
import time

import pandas as pd #1.3.5
import numpy as np
import sklearn.preprocessing as skpr
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report #1.0.0
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from tqdm import tqdm

import os
import json
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# -- MODELOS: --
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from  sklearn.ensemble import RandomForestClassifier



df = pd.read_csv('./DF4/df4.csv', sep=';') #El csv está separado por ';' en vez de por ',' como es por defecto.
df['Type'].value_counts()

/tmp/ipykernel_5143/4201451070.py:33: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./DF4/df4.csv', sep=';') #El csv está separado por ';' en vez de por ',' como es por defecto.


Type
Paciente    358843
Madre        24010
Padre        21565
Familia        209
Name: count, dtype: int64

In [2]:
#Separación del DataFrame en grupos:

#1 - 45K tweet with TEA/ASP patients and 45K tweets padre/madre/familia
t_pacientes= df.loc[(df["Type"]=="Paciente")]
t_familia= df.loc[(df["Type"]!="Paciente")]

data1_paciente = t_pacientes.sample(n=45000)
data1_familia = t_familia.sample(n=45000)


""" #TEA /// ASP
temp_TEA=t_pacientes.loc[(t_pacientes["Cate_1"] == "TEA") & (t_pacientes["Cate_2"] != "ASP")] 
temp_ASP=t_pacientes.loc[(t_pacientes["Cate_1"] == "ASP") & (t_pacientes["Cate_2"] != "TEA")]

print(len(temp_ASP))

data2_paciente_TEA = temp_TEA.sample(n=87000)
temp_n = len(temp_ASP)
data2_paciente_ASP = temp_ASP.sample(n=temp_n)

#TA /// TDAH
temp_TA=t_pacientes.loc[(t_pacientes["Cate_2"].isnull())] 
temp_TDAH=t_pacientes.loc[(t_pacientes["Cate_2"] == "TDAH") | (t_pacientes["Cate_3"] == "TDAH")]

data3_paciente_TA = temp_TA.sample(n=70000)
temp_n = len(temp_TDAH)
data3_paciente_TDAH = temp_TDAH.sample(n=temp_n) """



features = pd.concat([data1_paciente, data1_familia], ignore_index=True)
labels_aux = np.array(features['Type'])
labels = [1 if x == 'Paciente' else 0 for x in labels_aux]

print(features.head(5))
print("--------------------------")

features = features.drop('Type', axis = 1)
features = features.drop('Cate_1', axis = 1)
features = features.drop('Cate_2', axis = 1)
features = features.drop('Cate_3', axis = 1)
print(features.head(5))

       Type Cate_1 Cate_2 Cate_3  \
0  Paciente    TEA    NaN    NaN   
1  Paciente    TEA    NaN    NaN   
2  Paciente    ASP    NaN    NaN   
3  Paciente    TEA    NaN    NaN   
4  Paciente    TEA    NaN    NaN   

                                               tweet  
0  This is your Sunday evening reminder that ever...  
1  Podcast: On #AutisticJoy, Personality &amp; Be...  
2  Thank you, grocery store workers. \n\nCashiers...  
3  Copy edits for The Paper &amp; Hearts Society ...  
4  1.30am and can't sleep. Anybody else gets bout...  
--------------------------
                                               tweet
0  This is your Sunday evening reminder that ever...
1  Podcast: On #AutisticJoy, Personality &amp; Be...
2  Thank you, grocery store workers. \n\nCashiers...
3  Copy edits for The Paper &amp; Hearts Society ...
4  1.30am and can't sleep. Anybody else gets bout...


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer as tf_idf
from sklearn.feature_extraction.text import CountVectorizer

save_features=features
X_train, X_test, Y_train, Y_test=train_test_split(save_features['tweet'], labels, test_size= 0.25, random_state = 42)

tfidf = tf_idf(norm = None)
cv = CountVectorizer()

tfidf_sw = tf_idf(norm=None, stop_words='english')
cv_sw = CountVectorizer(stop_words='english')

label_encoder = skpr.LabelEncoder()

#Estos de Y los usamos para todos porque son los targets
Y_train_ = label_encoder.fit_transform(Y_train) 
Y_test_ = label_encoder.transform(Y_test)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

X_train_tfidf_sw = tfidf_sw.fit_transform(X_train)
X_test_tfidf_sw = tfidf_sw.transform(X_test)

X_train_cv_sw = cv_sw.fit_transform(X_train)
X_test_cv_sw = cv_sw.transform(X_test)

In [4]:
import numpy as np
from gensim.models import Word2Vec
import nltk

# Aseguramos la descarga del tokenizador
nltk.download('punkt')

# 1. Tokenizar los textos
train_tokens = [nltk.word_tokenize(text.lower()) for text in X_train]
test_tokens = [nltk.word_tokenize(text.lower()) for text in X_test]

# 2. Entrenar el modelo Word2Vec con parámetros por defecto
# Solo especificamos workers=4 según tu requerimiento.
# Por defecto: vector_size=100, window=5, min_count=5.
w2v_model = Word2Vec(sentences=train_tokens, workers=8)

# 3. Función para promediar vectores (usando el tamaño por defecto: 100)
def get_average_vector(tokens, model, size=100):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(size)
    return np.mean(vectors, axis=0)

# 4. Transformar los datos a embeddings promediados
X_train_emb = np.array([get_average_vector(t, w2v_model) for t in train_tokens])
X_test_emb = np.array([get_average_vector(t, w2v_model) for t in test_tokens])

[nltk_data] Downloading package punkt to /home/xeihtt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
from nltk.stem import PorterStemmer
import nltk

# Aseguramos los recursos necesarios
nltk.download('punkt')
nltk.download('punkt_tab') # Recomendado para las versiones más recientes de NLTK

stemmer = PorterStemmer()

def stem_tokenizer(text):
    # Tokeniza y aplica el algoritmo de Porter a cada palabra
    tokens = nltk.word_tokenize(text.lower()) # Añadimos lower() para mayor consistencia de las palabras (no tenemos en cuenta si mayúscula o minúscula)
    return [stemmer.stem(w) for w in tokens]

# Instanciamos CountVectorizer con tu función de stemming
# Al igual que antes, token_pattern=None es necesario cuando usas un tokenizer propio
cv_ps = CountVectorizer(tokenizer=stem_tokenizer, token_pattern=None) #<- Este de aquí no tiene el parámetro norm
tfidf_ps = tf_idf(tokenizer=stem_tokenizer, token_pattern = None, norm=None)

# Transformamos los datos
X_train_cv_ps = cv_ps.fit_transform(X_train)
X_test_cv_ps = cv_ps.transform(X_test)

X_train_tfidf_ps = tfidf_ps.fit_transform(X_train)
X_test_tfidf_ps = tfidf_ps.transform(X_test)

# Ejemplo de lo que verías: "running", "runs", "ran" -> todos se cuentan bajo el token "run"

[nltk_data] Downloading package punkt to /home/xeihtt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/xeihtt/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: DecisionTreeClassifier
</h1>

In [ ]:
dec_tree = DecisionTreeClassifier(random_state=42, max_depth=9, min_samples_leaf=2, min_samples_split=4)

start = time.perf_counter()
dec_tree.fit(X_train_cv, Y_train_)
end = time.perf_counter()

dec_tree_predict = dec_tree.predict(X_test_cv)
dec_tree_time = end-start

print(f"El entrenamiento de DecisionTreeClassifier es: {dec_tree_time}")
print(f"Accuracy: {accuracy_score(Y_test_, dec_tree_predict)} | F1 Score: {f1_score(Y_test_, dec_tree_predict)}")


El entrenamiento de DecisionTreeClassifier es: 0.6511482610003441
Accuracy: 0.6086666666666667 | F1 Score: 0.7178879241293133


In [47]:
dec_tree = DecisionTreeClassifier(random_state=42, max_depth=9, min_samples_leaf=2, min_samples_split=4)

start = time.perf_counter()
dec_tree.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

dec_tree_predict = dec_tree.predict(X_test_tfidf)
dec_tree_time = end-start

print(f"El entrenamiento de DecisionTreeClassifier es: {dec_tree_time}")
print(f"Accuracy: {accuracy_score(Y_test_, dec_tree_predict)} | F1 Score: {f1_score(Y_test_, dec_tree_predict)}")


#Con los parámetros por defecto:

#El entrenamiento de DecisionTreeClassifier es: 36.172373163999964
#Accuracy: 0.6964888888888889 | F1 Score: 0.7057860497178062


El entrenamiento de DecisionTreeClassifier es: 0.6492866380012856
Accuracy: 0.6086666666666667 | F1 Score: 0.7178879241293133


In [48]:
dec_tree = DecisionTreeClassifier(random_state=42, max_depth=9, min_samples_leaf=2, min_samples_split=4)

start = time.perf_counter()
dec_tree.fit(X_train_emb, Y_train_)
end = time.perf_counter()

dec_tree_predict = dec_tree.predict(X_test_emb)
dec_tree_time = end-start

print(f"El entrenamiento de DecisionTreeClassifier es: {dec_tree_time}")
print(f"Accuracy: {accuracy_score(Y_test_, dec_tree_predict)} | F1 Score: {f1_score(Y_test_, dec_tree_predict)}")

El entrenamiento de DecisionTreeClassifier es: 7.251137276000009
Accuracy: 0.6368888888888888 | F1 Score: 0.6235023041474654


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: XGB
</h1>

In [ ]:
xgb = XGBClassifier(random_state = 42, colsample_bytree=0.8, gamma=5, max_depth=5, min_child_weight=1, subsample=0.8)

start = time.perf_counter()
xgb.fit(X_train_cv, Y_train_)
end = time.perf_counter()

xgb_predict = xgb.predict(X_test_cv) #Parte del tiempo también está aqui, porque la celda entera son 3.4s y el entrenamiento 2.0
xgb_time = end-start

print(f"El entrenamiento de XGB es: {xgb_time}")
print(f"Accuracy: {accuracy_score(Y_test_, xgb_predict)} | F1 Score: {f1_score(Y_test_, xgb_predict)}")

El entrenamiento de XGB es: 2.0146063520005555
Accuracy: 0.7208888888888889 | F1 Score: 0.7403456545108741


In [ ]:
xgb = XGBClassifier(random_state = 42, colsample_bytree=0.8, gamma=5, max_depth=5, min_child_weight=1, subsample=0.8)

start = time.perf_counter()
xgb.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

xgb_predict = xgb.predict(X_test_tfidf) #Parte del tiempo también está aqui, porque la celda entera son 3.1s y el entrenamiento 1.9
xgb_time = end-start

print(f"El entrenamiento de XGB es: {xgb_time}")
print(f"Accuracy: {accuracy_score(Y_test_, xgb_predict)} | F1 Score: {f1_score(Y_test_, xgb_predict)}")


#Con los parámetros por defecto:

#El entrenamiento de XGB es: 4.298980682000092
#Accuracy: 0.7221333333333333 | F1 Score: 0.7420579255714168

El entrenamiento de XGB es: 1.9183299050000642
Accuracy: 0.7208888888888889 | F1 Score: 0.7403456545108741


In [ ]:
xgb = XGBClassifier(random_state = 42, colsample_bytree=0.8, gamma=5, max_depth=5, min_child_weight=1, subsample=0.8)

start = time.perf_counter()
xgb.fit(X_train_emb, Y_train_)
end = time.perf_counter()

xgb_predict = xgb.predict(X_test_emb)
xgb_time = end-start

print(f"El entrenamiento de XGB es: {xgb_time}")
print(f"Accuracy: {accuracy_score(Y_test_, xgb_predict)} | F1 Score: {f1_score(Y_test_, xgb_predict)}")

El entrenamiento de XGB es: 0.7646672979990399
Accuracy: 0.6974222222222223 | F1 Score: 0.7052558663087714


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: KNN
</h1>

In [ ]:
knn = KNeighborsClassifier(n_jobs=-1, n_neighbors=1) #Este no tiene el parámetro random_state

start = time.perf_counter()
knn.fit(X_train_cv, Y_train_) 

knn_predict = knn.predict(X_test_cv) #TODO: Importante, el KNN por algún motivo tiene el cuello de botella del tiempo en el predict no en el fit
end = time.perf_counter()
knn_time = end-start

print(f"El entrenamiento de KNN es: {knn_time}")
print(f"Accuracy: {accuracy_score(Y_test_, knn_predict)} | F1 Score: {f1_score(Y_test_, knn_predict)}")

#Con los parámetros por defecto:

#El entrenamiento de KNN es: 0.004698548999840568
#Accuracy: 0.6327111111111111 | F1 Score: 0.6370663153271849

#Midiendo solo el tiempo de fit:
#El entrenamiento de KNN es: 0.0047277369994844776
#Accuracy: 0.6394222222222222 | F1 Score: 0.6346647453505652


El entrenamiento de KNN es: 25.150838164999982
Accuracy: 0.6268444444444444 | F1 Score: 0.6282652970866909


In [9]:
knn = KNeighborsClassifier(n_jobs=-1, n_neighbors=1) #Este no tiene el parámetro random_state

start = time.perf_counter()
knn.fit(X_train_tfidf, Y_train_) 

knn_predict = knn.predict(X_test_tfidf) #TODO: Importante, el KNN por algún motivo tiene el cuello de botella del tiempo en el predict no en el fit
end = time.perf_counter()
knn_time = end-start

print(f"El entrenamiento de KNN es: {knn_time}")
print(f"Accuracy: {accuracy_score(Y_test_, knn_predict)} | F1 Score: {f1_score(Y_test_, knn_predict)}")

#Con los parámetros por defecto:

#El entrenamiento de KNN es: 0.005414055999608536
#Accuracy: 0.5834666666666667 | F1 Score: 0.6727425099518123

#Midiendo solo el tiempo de fit:
#El entrenamiento de KNN es: 0.006837103001089417
#Accuracy: 0.6063555555555555 | F1 Score: 0.6626804280763225

El entrenamiento de KNN es: 25.04593686599992
Accuracy: 0.6021777777777778 | F1 Score: 0.6586584296228502


In [16]:
knn = KNeighborsClassifier(n_jobs=-1, n_neighbors=1) #Este no tiene el parámetro random_state

start = time.perf_counter()
knn.fit(X_train_emb, Y_train_) 

#Ahora con Embedding no se da tanto cuello de botella aqui. La diferencia entre F1 y accuracy es por el TF-IDF
knn_predict = knn.predict(X_test_emb) #TODO: Importante, el KNN por algún motivo tiene el cuello de botella del tiempo en el predict no en el fit
end = time.perf_counter()
knn_time = end-start

print(f"El entrenamiento de KNN es: {knn_time}")
print(f"Accuracy: {accuracy_score(Y_test_, knn_predict)} | F1 Score: {f1_score(Y_test_, knn_predict)}")

#Con los parámetros por defecto:

#El entrenamiento de KNN es: 0.004409925999425468
#Accuracy: 0.6846666666666666 | F1 Score: 0.6948255838960815

#Midiendo solo el tiempo de fit:
#El entrenamiento de KNN es: 0.004522752000411856
#Accuracy: 0.6650222222222222 | F1 Score: 0.6660463467588285

El entrenamiento de KNN es: 1.6343275399995036
Accuracy: 0.6626666666666666 | F1 Score: 0.6631756456909559


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: LogisticRegression
</h1>

In [ ]:
#Medición de métricas de: LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=300) #Tuve que subir max iter de 100 a 300 para evitar el warning

start = time.perf_counter()
log_reg.fit(X_train_cv, Y_train_)
end = time.perf_counter()

log_reg_predict = log_reg.predict(X_test_cv)
log_reg_time = end-start

print(f"El entrenamiento de Logistic Regression es: {log_reg_time}")
print(f"Accuracy: {accuracy_score(Y_test_, log_reg_predict)} | F1 Score: {f1_score(Y_test_, log_reg_predict)}")

El entrenamiento de Logistic Regression es: 23.4159632410001
Accuracy: 0.7807111111111111 | F1 Score: 0.7843342949558528


In [35]:
#Medición de métricas de: LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=300) #Tuve que subir max iter de 100 a 300 para evitar el warning

start = time.perf_counter()
log_reg.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

log_reg_predict = log_reg.predict(X_test_tfidf)
log_reg_time = end-start

print(f"El entrenamiento de Logistic Regression es: {log_reg_time}")
print(f"Accuracy: {accuracy_score(Y_test_, log_reg_predict)} | F1 Score: {f1_score(Y_test_, log_reg_predict)}")

El entrenamiento de Logistic Regression es: 15.275444547000006
Accuracy: 0.7600444444444444 | F1 Score: 0.7573592198103456


In [36]:
#Medición de métricas de: LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=300) #Tuve que subir max iter de 100 a 300 para evitar el warning

start = time.perf_counter()
log_reg.fit(X_train_emb, Y_train_)
end = time.perf_counter()

log_reg_predict = log_reg.predict(X_test_emb)
log_reg_time = end-start

print(f"El entrenamiento de Logistic Regression es: {log_reg_time}")
print(f"Accuracy: {accuracy_score(Y_test_, log_reg_predict)} | F1 Score: {f1_score(Y_test_, log_reg_predict)}")

El entrenamiento de Logistic Regression es: 5.159379493000415
Accuracy: 0.6508 | F1 Score: 0.6722289433064953


In [17]:
#Medición de métricas de: LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=300) #Tuve que subir max iter de 100 a 300 para evitar el warning

start = time.perf_counter()
log_reg.fit(X_train_cv_sw, Y_train_)
end = time.perf_counter()

log_reg_predict = log_reg.predict(X_test_cv_sw)
log_reg_time = end-start

print(f"El entrenamiento de Logistic Regression es: {log_reg_time}")
print(f"Accuracy: {accuracy_score(Y_test_, log_reg_predict)} | F1 Score: {f1_score(Y_test_, log_reg_predict)}")

El entrenamiento de Logistic Regression es: 13.324609805000364
Accuracy: 0.7813333333333333 | F1 Score: 0.7858262232282779


In [20]:
#Medición de métricas de: LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=300) #Tuve que subir max iter de 100 a 300 para evitar el warning

start = time.perf_counter()
log_reg.fit(X_train_tfidf_sw, Y_train_)
end = time.perf_counter()

log_reg_predict = log_reg.predict(X_test_tfidf_sw)
log_reg_time = end-start

print(f"El entrenamiento de Logistic Regression es: {log_reg_time}")
print(f"Accuracy: {accuracy_score(Y_test_, log_reg_predict)} | F1 Score: {f1_score(Y_test_, log_reg_predict)}")

El entrenamiento de Logistic Regression es: 7.446994492000158
Accuracy: 0.7629333333333334 | F1 Score: 0.7600323915781897


In [7]:
#Medición de métricas de: LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=300) #Tuve que subir max iter de 100 a 300 para evitar el warning

start = time.perf_counter()
log_reg.fit(X_train_cv_ps, Y_train_)
end = time.perf_counter()

log_reg_predict = log_reg.predict(X_test_cv_ps)
log_reg_time = end-start

print(f"El entrenamiento de Logistic Regression es: {log_reg_time}")
print(f"Accuracy: {accuracy_score(Y_test_, log_reg_predict)} | F1 Score: {f1_score(Y_test_, log_reg_predict)}")

El entrenamiento de Logistic Regression es: 32.05718954999975
Accuracy: 0.8076888888888889 | F1 Score: 0.8103773171479907


In [8]:
#Medición de métricas de: LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=300) #Tuve que subir max iter de 100 a 300 para evitar el warning

start = time.perf_counter()
log_reg.fit(X_train_tfidf_ps, Y_train_)
end = time.perf_counter()

log_reg_predict = log_reg.predict(X_test_tfidf_ps)
log_reg_time = end-start

print(f"El entrenamiento de Logistic Regression es: {log_reg_time}")
print(f"Accuracy: {accuracy_score(Y_test_, log_reg_predict)} | F1 Score: {f1_score(Y_test_, log_reg_predict)}")

El entrenamiento de Logistic Regression es: 21.468392831000074
Accuracy: 0.7827555555555555 | F1 Score: 0.7819609242572932


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: SGD
</h1>

In [ ]:
#Medición de métricas de: Stochastic Gradient Descent

sgd = SGDClassifier(random_state=42, n_jobs=-1)

start = time.perf_counter()
sgd.fit(X_train_cv, Y_train_)
end = time.perf_counter()

sgd_predict = sgd.predict(X_test_cv)
sgd_time = end-start

print(f"El entrenamiento de SGD es: {sgd_time}")
print(f"Accuracy: {accuracy_score(Y_test_, sgd_predict)} | F1 Score: {f1_score(Y_test_, sgd_predict)}")

El entrenamiento de SGD es: 0.4804043169997385
Accuracy: 0.7810222222222222 | F1 Score: 0.78840455228688


In [ ]:
#Medición de métricas de: Stochastic Gradient Descent

sgd = SGDClassifier(random_state=42, n_jobs=-1)

start = time.perf_counter()
sgd.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

sgd_predict = sgd.predict(X_test_tfidf)
sgd_time = end-start

print(f"El entrenamiento de SGD es: {sgd_time}")
print(f"Accuracy: {accuracy_score(Y_test_, sgd_predict)} | F1 Score: {f1_score(Y_test_, sgd_predict)}")

El entrenamiento de SGD es: 0.49073959899942565
Accuracy: 0.7572 | F1 Score: 0.761513947701576


In [ ]:
#Medición de métricas de: Stochastic Gradient Descent

sgd = SGDClassifier(random_state=42, n_jobs=-1)

start = time.perf_counter()
sgd.fit(X_train_emb, Y_train_)
end = time.perf_counter()

sgd_predict = sgd.predict(X_test_emb)
sgd_time = end-start

print(f"El entrenamiento de SGD es: {sgd_time}")
print(f"Accuracy: {accuracy_score(Y_test_, sgd_predict)} | F1 Score: {f1_score(Y_test_, sgd_predict)}")

El entrenamiento de SGD es: 0.7118097880002097
Accuracy: 0.6487555555555555 | F1 Score: 0.6758541487223658


In [18]:
#Medición de métricas de: Stochastic Gradient Descent

sgd = SGDClassifier(random_state=42, n_jobs=-1)

start = time.perf_counter()
sgd.fit(X_train_cv_sw, Y_train_)
end = time.perf_counter()

sgd_predict = sgd.predict(X_test_cv_sw)
sgd_time = end-start

print(f"El entrenamiento de SGD es: {sgd_time}")
print(f"Accuracy: {accuracy_score(Y_test_, sgd_predict)} | F1 Score: {f1_score(Y_test_, sgd_predict)}")

El entrenamiento de SGD es: 0.26378677999855427
Accuracy: 0.7781777777777777 | F1 Score: 0.7866820532546908


In [21]:
#Medición de métricas de: Stochastic Gradient Descent

sgd = SGDClassifier(random_state=42, n_jobs=-1)

start = time.perf_counter()
sgd.fit(X_train_tfidf_sw, Y_train_)
end = time.perf_counter()

sgd_predict = sgd.predict(X_test_tfidf_sw)
sgd_time = end-start

print(f"El entrenamiento de SGD es: {sgd_time}")
print(f"Accuracy: {accuracy_score(Y_test_, sgd_predict)} | F1 Score: {f1_score(Y_test_, sgd_predict)}")

El entrenamiento de SGD es: 0.1947304669993173
Accuracy: 0.7582666666666666 | F1 Score: 0.7610596142863418


In [9]:
#Medición de métricas de: Stochastic Gradient Descent

sgd = SGDClassifier(random_state=42, n_jobs=-1)

start = time.perf_counter()
sgd.fit(X_train_cv_ps, Y_train_)
end = time.perf_counter()

sgd_predict = sgd.predict(X_test_cv_ps)
sgd_time = end-start

print(f"El entrenamiento de SGD es: {sgd_time}")
print(f"Accuracy: {accuracy_score(Y_test_, sgd_predict)} | F1 Score: {f1_score(Y_test_, sgd_predict)}")

El entrenamiento de SGD es: 0.5320797180002046
Accuracy: 0.802 | F1 Score: 0.8057215123631765


In [10]:
#Medición de métricas de: Stochastic Gradient Descent

sgd = SGDClassifier(random_state=42, n_jobs=-1)

start = time.perf_counter()
sgd.fit(X_train_tfidf_ps, Y_train_)
end = time.perf_counter()

sgd_predict = sgd.predict(X_test_tfidf_ps)
sgd_time = end-start

print(f"El entrenamiento de SGD es: {sgd_time}")
print(f"Accuracy: {accuracy_score(Y_test_, sgd_predict)} | F1 Score: {f1_score(Y_test_, sgd_predict)}")

El entrenamiento de SGD es: 0.49568764700006795
Accuracy: 0.7826666666666666 | F1 Score: 0.7906319575269738


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: SVC
</h1>

In [ ]:
svc = SVC(random_state=42)

start = time.perf_counter()
svc.fit(X_train_cv, Y_train_)
end = time.perf_counter()

svc_predict = svc.predict(X_test_cv)
svc_time = end-start

print(f"El entrenamiento de SVC es: {svc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, svc_predict)} | F1 Score: {f1_score(Y_test_, svc_predict)}")

El entrenamiento de SVC es: 1513.1147476899996
Accuracy: 0.7689333333333334 | F1 Score: 0.7816005040957782


In [ ]:
svc = SVC(random_state=42)

start = time.perf_counter()
svc.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

svc_predict = svc.predict(X_test_tfidf)
svc_time = end-start

print(f"El entrenamiento de SVC es: {svc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, svc_predict)} | F1 Score: {f1_score(Y_test_, svc_predict)}")

El entrenamiento de SVC es: 1585.7948841280004
Accuracy: 0.7776 | F1 Score: 0.7826238053866204


In [ ]:
svc = SVC(random_state=42)

start = time.perf_counter()
svc.fit(X_train_emb, Y_train_)
end = time.perf_counter()

svc_predict = svc.predict(X_test_emb)
svc_time = end-start

print(f"El entrenamiento de SVC es: {svc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, svc_predict)} | F1 Score: {f1_score(Y_test_, svc_predict)}")

El entrenamiento de SVC es: 398.817962051
Accuracy: 0.7171555555555555 | F1 Score: 0.7321097827917158


In [ ]:
svc = SVC(random_state=42)

start = time.perf_counter()
svc.fit(X_train_cv_sw, Y_train_)
end = time.perf_counter()

svc_predict = svc.predict(X_test_cv_sw)
svc_time = end-start

print(f"El entrenamiento de SVC es: {svc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, svc_predict)} | F1 Score: {f1_score(Y_test_, svc_predict)}")

El entrenamiento de SVC es: 1169.5709972090008
Accuracy: 0.7752444444444444 | F1 Score: 0.786326953141505


In [ ]:
svc = SVC(random_state=42)

start = time.perf_counter()
svc.fit(X_train_tfidf_sw, Y_train_)
end = time.perf_counter()

svc_predict = svc.predict(X_test_tfidf_sw)
svc_time = end-start

print(f"El entrenamiento de SVC es: {svc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, svc_predict)} | F1 Score: {f1_score(Y_test_, svc_predict)}")

El entrenamiento de SVC es: 927.3230642440012
Accuracy: 0.7778666666666667 | F1 Score: 0.7837112688246495


In [11]:
svc = SVC(random_state=42)

start = time.perf_counter()
svc.fit(X_train_cv_ps, Y_train_)
end = time.perf_counter()

svc_predict = svc.predict(X_test_cv_ps)
svc_time = end-start

print(f"El entrenamiento de SVC es: {svc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, svc_predict)} | F1 Score: {f1_score(Y_test_, svc_predict)}")

El entrenamiento de SVC es: 1182.1070778109997
Accuracy: 0.7987555555555556 | F1 Score: 0.8045242617855293


In [12]:
svc = SVC(random_state=42)

start = time.perf_counter()
svc.fit(X_train_tfidf_ps, Y_train_)
end = time.perf_counter()

svc_predict = svc.predict(X_test_tfidf_ps)
svc_time = end-start

print(f"El entrenamiento de SVC es: {svc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, svc_predict)} | F1 Score: {f1_score(Y_test_, svc_predict)}")

El entrenamiento de SVC es: 1746.7135440449993
Accuracy: 0.8038666666666666 | F1 Score: 0.8060305041536636


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: LinearSVC
</h1>

In [ ]:
lin_svc = LinearSVC(max_iter = 5000, dual = False, random_state=42) #He tenido que pasar de 1000 por defecto a 5000 con dual=False para evitar el warning

start = time.perf_counter()
lin_svc.fit(X_train_cv, Y_train_)
end = time.perf_counter()

linsvc_predict = lin_svc.predict(X_test_cv)
linsvc_time = end-start

print(f"El entrenamiento de LinearSVC es: {linsvc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, linsvc_predict)} | F1 Score: {f1_score(Y_test_, linsvc_predict)}")

El entrenamiento de LinearSVC es: 15.497677193999948
Accuracy: 0.7736888888888889 | F1 Score: 0.7730837789661319


In [ ]:
lin_svc = LinearSVC(max_iter = 5000, dual=False, random_state=42) #He tenido que pasar de 1000 por defecto a 5000 con dual=False para evitar el warning

start = time.perf_counter()
lin_svc.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

linsvc_predict = lin_svc.predict(X_test_tfidf)
linsvc_time = end-start

print(f"El entrenamiento de LinearSVC es: {linsvc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, linsvc_predict)} | F1 Score: {f1_score(Y_test_, linsvc_predict)}")

El entrenamiento de LinearSVC es: 184.16708314799996
Accuracy: 0.722 | F1 Score: 0.7190910315713837


In [ ]:
lin_svc = LinearSVC(max_iter = 5000, dual=False, random_state=42) 

start = time.perf_counter()
lin_svc.fit(X_train_emb, Y_train_)
end = time.perf_counter()

linsvc_predict = lin_svc.predict(X_test_emb)
linsvc_time = end-start

print(f"El entrenamiento de LinearSVC es: {linsvc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, linsvc_predict)} | F1 Score: {f1_score(Y_test_, linsvc_predict)}")

El entrenamiento de LinearSVC es: 6.747435957999983
Accuracy: 0.6519555555555555 | F1 Score: 0.6771121098420814


In [ ]:
lin_svc = LinearSVC(max_iter = 5000, dual = False, random_state=42) #He tenido que pasar de 1000 por defecto a 5000 con dual=False para evitar el warning

start = time.perf_counter()
lin_svc.fit(X_train_cv_sw, Y_train_)
end = time.perf_counter()

linsvc_predict = lin_svc.predict(X_test_cv_sw)
linsvc_time = end-start

print(f"El entrenamiento de LinearSVC es: {linsvc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, linsvc_predict)} | F1 Score: {f1_score(Y_test_, linsvc_predict)}")

El entrenamiento de LinearSVC es: 11.207513547999042
Accuracy: 0.7733333333333333 | F1 Score: 0.7724027133166727


In [ ]:
lin_svc = LinearSVC(max_iter = 5000, dual=False, random_state=42) #He tenido que pasar de 1000 por defecto a 5000 con dual=False para evitar el warning

start = time.perf_counter()
lin_svc.fit(X_train_tfidf_sw, Y_train_)
end = time.perf_counter()

linsvc_predict = lin_svc.predict(X_test_tfidf_sw)
linsvc_time = end-start

print(f"El entrenamiento de LinearSVC es: {linsvc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, linsvc_predict)} | F1 Score: {f1_score(Y_test_, linsvc_predict)}")

El entrenamiento de LinearSVC es: 94.91053582600034
Accuracy: 0.7189777777777778 | F1 Score: 0.7138525591709282


In [6]:
lin_svc = LinearSVC(max_iter = 5000, dual = False, random_state=42) #He tenido que pasar de 1000 por defecto a 5000 con dual=False para evitar el warning

start = time.perf_counter()
lin_svc.fit(X_train_cv_ps, Y_train_)
end = time.perf_counter()

linsvc_predict = lin_svc.predict(X_test_cv_ps)
linsvc_time = end-start

print(f"El entrenamiento de LinearSVC es: {linsvc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, linsvc_predict)} | F1 Score: {f1_score(Y_test_, linsvc_predict)}")

El entrenamiento de LinearSVC es: 18.72969737699998
Accuracy: 0.7937333333333333 | F1 Score: 0.7947822241874862


In [7]:
lin_svc = LinearSVC(max_iter = 5000, dual=False, random_state=42) #He tenido que pasar de 1000 por defecto a 5000 con dual=False para evitar el warning

start = time.perf_counter()
lin_svc.fit(X_train_tfidf_ps, Y_train_)
end = time.perf_counter()

linsvc_predict = lin_svc.predict(X_test_tfidf_ps)
linsvc_time = end-start

print(f"El entrenamiento de LinearSVC es: {linsvc_time}")
print(f"Accuracy: {accuracy_score(Y_test_, linsvc_predict)} | F1 Score: {f1_score(Y_test_, linsvc_predict)}")

El entrenamiento de LinearSVC es: 238.26188226099998
Accuracy: 0.7408888888888889 | F1 Score: 0.7389162561576355


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: MLPClassifier
</h1>

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=400)

start = time.perf_counter()
mlp.fit(X_train_cv, Y_train_)
end = time.perf_counter()

# Predicciones y métricas
mlp_predict = mlp.predict(X_test_cv)
mlp_time = end - start

print(f"El entrenamiento de MLPClassifier es: {mlp_time}")
print(f"Accuracy: {accuracy_score(Y_test_, mlp_predict)} | F1 Score: {f1_score(Y_test_, mlp_predict)}")

El entrenamiento de MLPClassifier es: 5153.523013530999
Accuracy: 0.7765777777777778 | F1 Score: 0.7763491569159585


In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=400)

start = time.perf_counter()
mlp.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

# Predicciones y métricas
mlp_predict = mlp.predict(X_test_tfidf)
mlp_time = end - start

print(f"El entrenamiento de MLPClassifier es: {mlp_time}")
print(f"Accuracy: {accuracy_score(Y_test_, mlp_predict)} | F1 Score: {f1_score(Y_test_, mlp_predict)}")

El entrenamiento de MLPClassifier es: 1730.5103423229993
Accuracy: 0.7783555555555556 | F1 Score: 0.7830512898594858


In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=400)

start = time.perf_counter()
mlp.fit(X_train_emb, Y_train_)
end = time.perf_counter()

# Predicciones y métricas
mlp_predict = mlp.predict(X_test_emb)
mlp_time = end - start

print(f"El entrenamiento de MLPClassifier es: {mlp_time}")
print(f"Accuracy: {accuracy_score(Y_test_, mlp_predict)} | F1 Score: {f1_score(Y_test_, mlp_predict)}")

El entrenamiento de MLPClassifier es: 56.84932566799989
Accuracy: 0.7151111111111111 | F1 Score: 0.716797737916409


In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=400)

start = time.perf_counter()
mlp.fit(X_train_cv_sw, Y_train_)
end = time.perf_counter()

# Predicciones y métricas
mlp_predict = mlp.predict(X_test_cv_sw)
mlp_time = end - start

print(f"El entrenamiento de MLPClassifier es: {mlp_time}")
print(f"Accuracy: {accuracy_score(Y_test_, mlp_predict)} | F1 Score: {f1_score(Y_test_, mlp_predict)}")

El entrenamiento de MLPClassifier es: 5282.842938874999
Accuracy: 0.7664 | F1 Score: 0.7647480082356101


In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=400)

start = time.perf_counter()
mlp.fit(X_train_tfidf_sw, Y_train_)
end = time.perf_counter()

# Predicciones y métricas
mlp_predict = mlp.predict(X_test_tfidf_sw)
mlp_time = end - start

print(f"El entrenamiento de MLPClassifier es: {mlp_time}")
print(f"Accuracy: {accuracy_score(Y_test_, mlp_predict)} | F1 Score: {f1_score(Y_test_, mlp_predict)}")

El entrenamiento de MLPClassifier es: 2686.800531724999
Accuracy: 0.7763555555555556 | F1 Score: 0.7760170924953262


In [8]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=400)

start = time.perf_counter()
mlp.fit(X_train_cv_ps, Y_train_)
end = time.perf_counter()

# Predicciones y métricas
mlp_predict = mlp.predict(X_test_cv_ps)
mlp_time = end - start

print(f"El entrenamiento de MLPClassifier es: {mlp_time}")
print(f"Accuracy: {accuracy_score(Y_test_, mlp_predict)} | F1 Score: {f1_score(Y_test_, mlp_predict)}")

El entrenamiento de MLPClassifier es: 5082.141805677
Accuracy: 0.8066666666666666 | F1 Score: 0.8075050889459244


In [9]:
mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42, max_iter=400)

start = time.perf_counter()
mlp.fit(X_train_tfidf_ps, Y_train_)
end = time.perf_counter()

# Predicciones y métricas
mlp_predict = mlp.predict(X_test_tfidf_ps)
mlp_time = end - start

print(f"El entrenamiento de MLPClassifier es: {mlp_time}")
print(f"Accuracy: {accuracy_score(Y_test_, mlp_predict)} | F1 Score: {f1_score(Y_test_, mlp_predict)}")

El entrenamiento de MLPClassifier es: 1707.8608244259995
Accuracy: 0.7992888888888889 | F1 Score: 0.8004242531377055


<h1 style="text-align: center; color: #ffffff; background-color: #2e86de; padding: 20px; border-radius: 10px;">
    Pruebas del modelo: RandomForestClassifier
</h1>

In [ ]:
rf = RandomForestClassifier(random_state=42)

start = time.perf_counter()
rf.fit(X_train_cv, Y_train_)
end = time.perf_counter()

rf_predict = rf.predict(X_test_cv)
rf_time = end-start

print(f"El entrenamiento de RandomForestClassifier es: {rf_time}")
print(f"Accuracy: {accuracy_score(Y_test_, rf_predict)} | F1 Score: {f1_score(Y_test_, rf_predict)}")

El entrenamiento de RandomForestClassifier es: 334.2845553720001
Accuracy: 0.7572 | F1 Score: 0.7793351375368582


In [ ]:
rf = RandomForestClassifier(random_state=42)

start = time.perf_counter()
rf.fit(X_train_tfidf, Y_train_)
end = time.perf_counter()

rf_predict = rf.predict(X_test_tfidf)
rf_time = end-start

print(f"El entrenamiento de RandomForestClassifier es: {rf_time}")
print(f"Accuracy: {accuracy_score(Y_test_, rf_predict)} | F1 Score: {f1_score(Y_test_, rf_predict)}")

El entrenamiento de RandomForestClassifier es: 331.212113673
Accuracy: 0.7605333333333333 | F1 Score: 0.7810290173128506


In [ ]:
rf = RandomForestClassifier(random_state=42)

start = time.perf_counter()
rf.fit(X_train_emb, Y_train_)
end = time.perf_counter()

rf_predict = rf.predict(X_test_emb)
rf_time = end-start

print(f"El entrenamiento de RandomForestClassifier es: {rf_time}")
print(f"Accuracy: {accuracy_score(Y_test_, rf_predict)} | F1 Score: {f1_score(Y_test_, rf_predict)}")

El entrenamiento de RandomForestClassifier es: 81.44095093400028
Accuracy: 0.7013777777777778 | F1 Score: 0.7096746316380763


In [ ]:
rf = RandomForestClassifier(random_state=42)

start = time.perf_counter()
rf.fit(X_train_cv_sw, Y_train_)
end = time.perf_counter()

rf_predict = rf.predict(X_test_cv_sw)
rf_time = end-start

print(f"El entrenamiento de RandomForestClassifier es: {rf_time}")
print(f"Accuracy: {accuracy_score(Y_test_, rf_predict)} | F1 Score: {f1_score(Y_test_, rf_predict)}")

El entrenamiento de RandomForestClassifier es: 336.08431354100003
Accuracy: 0.7626222222222222 | F1 Score: 0.7791606367583213


In [ ]:
rf = RandomForestClassifier(random_state=42)

start = time.perf_counter()
rf.fit(X_train_tfidf_sw, Y_train_)
end = time.perf_counter()

rf_predict = rf.predict(X_test_tfidf_sw)
rf_time = end-start

print(f"El entrenamiento de RandomForestClassifier es: {rf_time}")
print(f"Accuracy: {accuracy_score(Y_test_, rf_predict)} | F1 Score: {f1_score(Y_test_, rf_predict)}")

El entrenamiento de RandomForestClassifier es: 331.4512920709999
Accuracy: 0.7657333333333334 | F1 Score: 0.7812953819343595


In [10]:
rf = RandomForestClassifier(random_state=42)

start = time.perf_counter()
rf.fit(X_train_cv_ps, Y_train_)
end = time.perf_counter()

rf_predict = rf.predict(X_test_cv_ps)
rf_time = end-start

print(f"El entrenamiento de RandomForestClassifier es: {rf_time}")
print(f"Accuracy: {accuracy_score(Y_test_, rf_predict)} | F1 Score: {f1_score(Y_test_, rf_predict)}")

El entrenamiento de RandomForestClassifier es: 325.025196812001
Accuracy: 0.7780444444444444 | F1 Score: 0.7934827557687536


In [11]:
rf = RandomForestClassifier(random_state=42)

start = time.perf_counter()
rf.fit(X_train_tfidf_ps, Y_train_)
end = time.perf_counter()

rf_predict = rf.predict(X_test_tfidf_ps)
rf_time = end-start

print(f"El entrenamiento de RandomForestClassifier es: {rf_time}")
print(f"Accuracy: {accuracy_score(Y_test_, rf_predict)} | F1 Score: {f1_score(Y_test_, rf_predict)}")

El entrenamiento de RandomForestClassifier es: 328.3676471559993
Accuracy: 0.7780444444444444 | F1 Score: 0.7934144121783735
